In [1]:
# Load env variables and create client
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
model = "google/gemini-2.5-flash-lite"

In [2]:
# Web search tool schema
# - max_uses: limits the model to at most 5 web searches per request
# - allowed_domains: restricts searches to evidence-based medical/fitness sources
#   so answers come from peer-reviewed literature (PubMed/NIH) instead of random blogs
web_search_tool = {
    "type": "web_search_preview",
    "search_context_size": "medium",
    "max_uses": 5,
    "allowed_domains": [
        "pubmed.ncbi.nlm.nih.gov",
        "www.ncbi.nlm.nih.gov",
        "nih.gov",
        "medlineplus.gov",
        "mayoclinic.org",
        "health.harvard.edu",
        "who.int",
        "cdc.gov"
    ]
}

tools = [web_search_tool]
print("Web search tool configured with:")
print(f"  max_uses       : {web_search_tool['max_uses']}")
print(f"  allowed_domains: {web_search_tool['allowed_domains']}")

Web search tool configured with:
  max_uses       : 5
  allowed_domains: ['pubmed.ncbi.nlm.nih.gov', 'www.ncbi.nlm.nih.gov', 'nih.gov', 'medlineplus.gov', 'mayoclinic.org', 'health.harvard.edu', 'who.int', 'cdc.gov']


In [3]:
# Helper: run a single web-search-enabled chat request
def search_chat(user_query, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_query})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
    )
    return response

In [4]:
# Example 1: medical advice (searches restricted to NIH / PubMed / trusted health domains)
system_prompt = (
    "You are a health information assistant. "
    "Always cite the source URL when you use web search results. "
    "Only rely on peer-reviewed or government health sources."
)

query = "What does current evidence say about the health benefits of omega-3 fatty acids?"

response = search_chat(query, system=system_prompt)
print(response.choices[0].message.content)

None


In [5]:
# Example 2: exercise science advice
query = (
    "How many minutes of moderate aerobic exercise per week does current research "
    "recommend for adults to reduce cardiovascular disease risk?"
)

response = search_chat(query, system=system_prompt)
print(response.choices[0].message.content)

None


In [6]:
# Inspect the raw response to see tool usage and citations
import json

query = "What are the CDC guidelines on vitamin D supplementation?"
response = search_chat(query, system=system_prompt)

choice = response.choices[0]
print("finish_reason:", choice.finish_reason)
print("\n--- Answer ---")
print(choice.message.content)

# Print any tool_calls metadata if present
if hasattr(choice.message, "tool_calls") and choice.message.tool_calls:
    print("\n--- Tool calls used ---")
    for tc in choice.message.tool_calls:
        print(json.dumps(json.loads(tc.function.arguments), indent=2))

finish_reason: stop

--- Answer ---
I found information on vitamin D supplementation from the CDC.

For infants and toddlers, the CDC recommends:
*   **Children younger than 12 months:** 400 international units (IU) of vitamin D each day.
*   **Children 12 to 24 months:** 600 IU of vitamin D each day.

These recommendations are important for bone health and to prevent rickets, a condition that softens bones.

For breastfed infants, specifically, the CDC notes that breast milk alone does not provide enough vitamin D. Therefore, breastfed and partially breastfed infants should receive a vitamin D supplement of 400 IU daily starting shortly after birth. Formula-fed babies typically do not need a supplement if they consume 32 ounces of infant formula or more per day, as infant formula is fortified with vitamin D.

The CDC also highlights that certain populations are at higher risk for vitamin D deficiency, including non-Hispanic Black Americans.

For more detailed information, you can refe